In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# =========================
# LOAD DATA
# =========================
df = pd.read_csv('../data/fake_job_postings.csv')
df.fillna('', inplace=True)

df['text'] = df['title'] + " " + df['description'] + " " + df['requirements']

X = df['text']
y = df['fraudulent']

# =========================
# BETTER TF-IDF 
# =========================
vectorizer = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,2),
    stop_words='english'
)

X_vec = vectorizer.fit_transform(X)

# =========================
# TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# MODEL 1: Logistic Regression
# =========================
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_lr))

from sklearn.metrics import classification_report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

# =========================
# MODEL 2: Naive Bayes 
# =========================
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

# =========================
# BEST MODEL SELECT
# =========================
best_model = nb if accuracy_score(y_test, y_pred_nb) > accuracy_score(y_test, y_pred_lr) else lr

print("Best Model:", type(best_model).__name__)

# =========================
# SAVE MODEL
# =========================
joblib.dump(best_model, '../models/model.pkl')
joblib.dump(vectorizer, '../models/vectorizer.pkl')

print(" Model saved successfully")

Logistic Accuracy: 0.9611297539149888

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3403
           1       0.57      0.84      0.68       173

    accuracy                           0.96      3576
   macro avg       0.78      0.90      0.83      3576
weighted avg       0.97      0.96      0.96      3576

Naive Bayes Accuracy: 0.9625279642058165
Best Model: MultinomialNB
 Model saved successfully
